# 01 — Offline sandbox environment setup

Prepare the **local-mailroom-sandbox** for offline work:

1. Ensure `.env` exists (from `config/.env.example`)
2. Activate an Ollama (or other) profile and write `data/runtime/taxonomy.yaml`
3. Verify Dockerfile, Compose, fixtures, and notebooks are present

Run this notebook **before** data cleaning (`02_…`) and smoke (`03_…`).

Docker path: `sandbox up --compose-profile jupyter` → http://127.0.0.1:8888/lab  
Docs: `docs/docker-offline.md`

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Prefer the mounted / editable checkout when running in Docker or a venv.
ROOT = Path(os.environ.get("SANDBOX_ROOT") or Path.cwd())
if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
    os.environ.setdefault("SANDBOX_ROOT", str(ROOT))

from mailroom_sandbox.prep import ensure_dotenv_from_example, environment_checklist
from mailroom_sandbox.paths import repo_root

print("repo_root =", repo_root())

## Dotenv

Never overwrites an existing `.env`. Inside Compose, sibling URLs (`ollama`, `langfuse-web`) are injected by the `jupyter` service.

In [ ]:
env_path = ensure_dotenv_from_example()
print("env →", env_path)
assert env_path is not None and env_path.is_file(), "config/.env.example missing"

## Activate profile + checklist

In [ ]:
PROFILE = os.environ.get("SANDBOX_PROFILE", "ollama")
check = environment_checklist(PROFILE)
print(json.dumps(check, indent=2))
assert check["dockerfile_ok"], "deploy/Dockerfile missing"
assert check["compose_file_ok"], "deploy/docker-compose.yml missing"
assert check["fixtures_manifest_ok"], "data/fixtures/manifest.csv missing"
assert check["activation_ok"], check.get("activation_error")
assert check["notebooks"], "expected notebooks under notebooks/"
print("\noffline_ready =", check["offline_ready"])

## Optional: probe local providers

Skipped unless services are up. Mock evals in notebook 03 do not need this.

In [ ]:
from mailroom_sandbox.health import health_check

try:
    health = health_check(PROFILE)
    print(json.dumps(health, indent=2, default=str))
except Exception as exc:  # noqa: BLE001
    print(f"health probe skipped/failed (ok for offline prep): {type(exc).__name__}: {exc}")

## Next

Open **`02_load_clean_prepare_data.ipynb`** to materialize cleaned JSONL under `data/runtime/prepared/`.